# 🔬 burst_scan — move explorer: whole runs, where they really start, and when you could know

A hands-on companion to `move_study.ipynb`. It loads **a few chosen stocks** (fast to re-run) and studies their
**runs**: moves of at least `MOVE_PCT`, whose first `MOVE_PCT` came within `WINDOW_MIN` of where they started.

**Why runs, not "2% slices".** Looking for "any 2% within 4 hours" finds the *last* 2% of a move, not its start: on
these names the slice's start had usually already moved 1.2% from the real bottom (half of them ≥ 1%, a third ≥ 2%
— the second 2% of a longer run). So each run is traced whole, and two moments in it are kept apart:

| moment | what it is | known when? | what it is good for |
|---|---|---|---|
| **onset** | the swing low (up run) / high (down run) the run starts from | only afterwards | *understanding* what came before a run |
| **confirmation** | the first bar whose **close** is `CONFIRM_PCT` past the recent low / high | at that bar's close — live | *trading*: entry here, stop behind the low, target the run |

| § | what |
|---|---|
| 0 | the knobs |
| 1 | **the runs** — how they are found, how many, how big; and how far into them the old 2% slices started |
| 2 | anatomy — run size, onset time of day, onset → confirmation → `MOVE_PCT` |
| 3 | **before the onset** — features at the onset and `LEAD_MIN` before it, against ordinary moments |
| 4 | **the confirmation** — every move that got `CONFIRM_PCT` off a low (or high): which ran, which failed, which features tell them apart, and what the trade from there earned — the honest, tradable test |
| 5 | the build-up — volume and range bar by bar around the onset and around the confirmation |
| 6 | **run cards** — each run drawn whole, with the session before it |
| 7 | checks and the scorecard |

**How a run is found.** The price is traced as alternating swings: a swing continues while it makes new highs (lows)
and ends only when price pulls back `REVERSAL_PCT` from its extreme. A **run** is a swing of at least `MOVE_PCT`
whose first `MOVE_PCT` came within `WINDOW_MIN` of its onset — one rally is one run, however long it goes on. By
default a run stays inside one session (`ALLOW_OVERNIGHT = False`).

**The stock picks** (hard-coded below, change them freely): GLW, ORCL and ALB — chosen on 2026-09-28 from the move
study's occurrence model on all 200 Morning200 names: the names whose coming moves it ranked best (within-month AUC
0.78 / 0.77 / 0.77) among those with a middling event rate, price ≥ $10 and ≥ $100M a day, from three industries.
In-sample: a good place to look, not evidence. Training window only; the holdout (May–June 2026) stays unread.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import backtest as bt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.data import resample_panel
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds every computation the five notebooks share; signals() never imports it.
_spec = importlib.util.spec_from_file_location(
    "burst_research", strategies.STRATEGIES_DIR / "burst_scan" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
strat, S = rs.STRAT, rs.S  # the loaded strategy and its module's functions
compose = S["compose"]
SIDES = ("long", "short")
COLOR = {"long": plots._LONG, "short": plots._SHORT}


def below(fig, y=-0.1):
    """Legend under the plot: the default top-right legend sits on the subplot titles."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


from trading_simple.data import load_panel

# --- 0 · the knobs --------------------------------------------------------------------------------
SYMBOLS = ["GLW", "ORCL", "ALB"]  # add or replace freely — only these are loaded
MOVE_PCT = 0.02  # a run is at least this big ...
WINDOW_MIN = 240  # ... and its first MOVE_PCT came within this many minutes of its onset
REVERSAL_PCT = 0.01  # a pullback of this much ends a run (smaller = more, shorter runs)
CONFIRM_PCT = 0.005  # the live trigger: a close this far off the recent low (high)
LOOKBACK_MIN = 120  # "recent" low / high for a confirmation: the last this many minutes
PRE_MIN = 60  # window of the "into the move" features (ret_pre, pre_range_vs_tod)
LEAD_MIN = 30  # features are also read this many minutes BEFORE the onset
CARD_PRE_MIN = 390  # run cards draw this many minutes before the onset (390 = a session)
POST_MIN = 30  # ... and this many after the run's end
BAR_MIN = 5  # candle size in minutes: 1 = the stored minutes, 5 / 10 / 15 = derived
PROFILE_DAYS = 10  # "normal for this time of day" = this many previous sessions ...
PROFILE_STAT = "mean"  # ... their average ("mean") or "median" (robust to one wild day)
SHORT_MIN = 15  # window of the short-term volume / range features
ATR_DAYS = 14  # daily ATR smoothing, in sessions
ALLOW_OVERNIGHT = False  # True: runs (and confirmations) may cross the close
SKIP_OPEN_MIN = 15  # §3-§6 leave out onsets / confirmations in the first N minutes: that is the
# overnight gap resolving, with no intraday "before" (§1-§2 count everything)
START, END = "2025-06-18", "2026-04-30"  # the training window; the holdout stays unread
RUNS_PER_STOCK = 4  # run cards per stock (half up, half down)
RUN_PICK = {}  # draw specific runs instead, e.g. {"GLW": [3, 17]} (numbers from the §1 table)
SEED = 0

## 1 · The runs

In [ ]:
panel = load_panel(
    engine, SYMBOLS, "1m", START, END, session="regular", exchanges=settings.exchanges
)
if BAR_MIN > 1:
    panel = resample_panel(panel, BAR_MIN)
tz = panel.tz
codes, bar_no = S["_sessions"](panel.close.index, tz)
mpb = S["_minutes_per_bar"](panel.bar)
n_bars = panel.close.shape[0]
warm = max(PROFILE_DAYS, ATR_DAYS) + 1  # sessions before the baselines exist
o_, h_, l_, c_, v_ = (
    x.to_numpy(float) for x in (panel.open_, panel.high, panel.low, panel.close, panel.volume)
)

runs_all = rs.swing_runs(
    panel,
    reversal_pct=REVERSAL_PCT,
    min_run_pct=MOVE_PCT,
    max_run_min=WINDOW_MIN,
    confirm_pct=CONFIRM_PCT,
    allow_overnight=ALLOW_OVERNIGHT,
)
runs_all = runs_all[codes[runs_all["onset"]] >= warm].reset_index(drop=True)
runs_all["side"] = np.where(runs_all["direction"] > 0, "up", "down")
runs_all["run"] = runs_all.groupby("symbol").cumcount()
runs_all["local"] = panel.close.index[runs_all["onset"]].tz_convert(tz)
runs_all["at_open"] = bar_no[runs_all["onset"]] * mpb < SKIP_OPEN_MIN
runs = runs_all[~runs_all["at_open"]].reset_index(drop=True)  # what §3-§6 study
n_sess = int(codes.max()) + 1 - warm
print(
    f"{n_bars:,} {panel.bar} bars · {len(SYMBOLS)} stocks · {n_sess} sessions after the "
    f"{warm}-session warm-up · runs ≥ {MOVE_PCT:.1%}, first {MOVE_PCT:.1%} within {WINDOW_MIN} "
    f"min, ended by a {REVERSAL_PCT:.1%} pullback"
)
summary = runs_all.groupby("symbol").agg(
    runs=("run", "size"),
    up=("direction", lambda d: (d > 0).sum()),
    down=("direction", lambda d: (d < 0).sum()),
    median_size=("size", "median"),
    median_min_to_move=("to_hit_min", "median"),
    median_duration_min=("duration_min", "median"),
    onset_at_open=("at_open", "mean"),
)
summary["per session"] = summary["runs"] / n_sess
summary["studied (§3-§6)"] = runs.groupby("symbol").size()
display(summary.round(3))

# How far into these runs did the old "any 2% within 4 h" slices start?
old = rs.move_episodes(
    panel, move_pct=MOVE_PCT, window_min=WINDOW_MIN, allow_overnight=ALLOW_OVERNIGHT
)
old = old[(codes[old["start"]] >= warm) & (bar_no[old["start"]] * mpb >= SKIP_OPEN_MIN)]
already = []
for r in old.itertuples():
    first = r.start - bar_no[r.start]
    lo_i = max(first, r.start - S["_bars"](WINDOW_MIN, panel.bar))
    seg = l_[lo_i : r.start + 1, r.col] if r.direction > 0 else h_[lo_i : r.start + 1, r.col]
    ext = np.nanmin(seg) if r.direction > 0 else np.nanmax(seg)
    already.append(r.anchor / ext - 1 if r.direction > 0 else 1 - r.anchor / ext)
already = pd.Series(already)
print(
    f"the old 2% slices had already moved, from the prior {WINDOW_MIN} min's extreme: median "
    f"{already.median():.2%}; ≥ 1% for {np.mean(already >= 0.01):.0%}, ≥ {MOVE_PCT:.0%} for "
    f"{np.mean(already >= MOVE_PCT):.0%} of them — runs start at the extreme itself."
)
runs_all[
    [
        "symbol",
        "run",
        "local",
        "side",
        "size",
        "to_confirm_min",
        "to_hit_min",
        "duration_min",
        "at_open",
    ]
].head(10).round(4)

## 2 · Anatomy (Fig 2)

**Top left** how big the runs grow (every run clears `MOVE_PCT`; how far past it?). **Top right** when they start
(minutes after the open) — the cluster in the first minutes is the overnight gap resolving. **Bottom** how long from
the onset to the confirmation (the live trigger) and to `MOVE_PCT`: a confirmation that comes late leaves little of
the run to trade.

In [ ]:
fig = make_subplots(
    rows=2,
    cols=2,
    vertical_spacing=0.16,
    subplot_titles=(
        "run size (%)",
        "onset: minutes after the open",
        "onset → confirmation (min)",
        f"onset → {MOVE_PCT:.0%} reached (min)",
    ),
)
for side, colr in (("up", plots._LONG), ("down", plots._SHORT)):
    d = runs_all[runs_all["side"] == side]
    kw = {"name": side, "marker_color": colr, "opacity": 0.65}
    fig.add_trace(go.Histogram(x=100 * d["size"], xbins={"size": 0.25}, **kw), row=1, col=1)
    fig.add_trace(
        go.Histogram(x=bar_no[d["onset"]] * mpb, xbins={"size": 15}, showlegend=False, **kw),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Histogram(x=d["to_confirm_min"], xbins={"size": mpb}, showlegend=False, **kw),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Histogram(x=d["to_hit_min"], xbins={"size": 10}, showlegend=False, **kw), row=2, col=2
    )
fig.update_layout(barmode="overlay")
plots._base(
    fig,
    title="Fig 2 · the runs: size, start, and the time to confirm and to reach the move",
    height=620,
)
below(fig, -0.08)
fig.show()
late = runs_all["confirm_after_hit"].mean()
print(
    f"runs confirmed only once {MOVE_PCT:.0%} was already reached (one explosive bar): {late:.0%}"
)

## 3 · Before the onset (Fig 3)

Every feature is known at the **close** of the bar it is read on. "Normal for this time of day" is the
`PROFILE_STAT` over the previous `PROFILE_DAYS` sessions at the same clock time (the current session excluded).

| feature | meaning |
|---|---|
| `vol_vs_tod`, `range_vs_tod` | volume / high–low range over the last `SHORT_MIN` minutes ÷ their normal for those clock times |
| `bar_vol_vs_tod`, `bar_range_vs_tod` | the same for the bar alone |
| `cum_vol_vs_tod` | the session's volume so far ÷ its normal by this time (a busy day?) |
| `pre_range_vs_tod` | range over the last `PRE_MIN` minutes ÷ its normal (below 1 = compressed, a "squeeze") |
| `atr_daily_pct`, `atr_intraday_pct` | daily ATR (known at the open) and 14-bar ATR, ÷ price |
| `ret_pre`, `ret_session`, `gap_pct` | return over the last `PRE_MIN` minutes, since the open; today's gap |
| `vwap_dist_atr`, `day_pos` | distance from the session VWAP (daily ATRs); place in today's range (0 low … 1 high) |
| `to_prev_high_atr`, `to_prev_low_atr` | distance to yesterday's high / low, in daily ATRs |
| `ret_5d`, `ret_20d` | the last 5 / 20 sessions' return |
| `prev_day_ret`, `prev_day_vol_vs_avg`, `prev_day_range_atr` | yesterday: its return, its volume ÷ the 20-day average, its range ÷ ATR |
| `minutes_since_open`, `minutes_to_close` | the clock |

The comparison group is **ordinary moments**: bars of the same stock and half hour from which no `MOVE_PCT` move came
within `WINDOW_MIN` (two per run). Three panels — AUC run vs ordinary **at the onset** and **`LEAD_MIN` before it**,
and up vs down `LEAD_MIN` before it. 0.5 = no information.

**⚠ Direction before an onset is partly built in.** An onset is a swing extreme, and a swing low is *by definition*
the end of a decline — so "the price had been falling" before an up run is the definition, not a discovery. The
honest direction test is §4.

In [ ]:
feats = rs.explore_features(
    panel,
    profile_days=PROFILE_DAYS,
    stat=PROFILE_STAT,
    atr_days=ATR_DAYS,
    short_minutes=SHORT_MIN,
    pre_minutes=PRE_MIN,
)
FEAT_NAMES = list(feats)
lead_b = S["_bars"](LEAD_MIN, panel.bar)
q = rs.quiet_controls(
    panel,
    runs.rename(columns={"onset": "start"}),
    move_pct=MOVE_PCT,
    window_min=WINDOW_MIN,
    seed=SEED,
)
q = q[codes[q["start"]] >= warm].reset_index(drop=True)
lead_rows = np.maximum(runs["onset"] - lead_b, 0)
same = np.asarray(codes[lead_rows] == codes[runs["onset"]])  # the lead stays in the session
F_on = rs.at_points(feats, runs["onset"], runs["col"]).assign(
    symbol=runs["symbol"], side=runs["side"]
)
F_lead = rs.at_points(feats, lead_rows, runs["col"]).assign(
    symbol=runs["symbol"], side=runs["side"]
)[same]
q_lead = np.maximum(q["start"] - lead_b, 0)
F_q = rs.at_points(feats, q["start"], q["col"]).assign(symbol=q["symbol"], side="ordinary")
F_ql = rs.at_points(feats, q_lead, q["col"]).assign(symbol=q["symbol"], side="ordinary")[
    np.asarray(codes[q_lead] == codes[q["start"]])
]
print(f"{len(runs)} runs studied, {len(q)} ordinary moments")


def feature_auc(a, b):
    """AUC of each feature for 'a row from a' vs 'a row from b' (NaN rows dropped); 0.5 = none."""
    y = np.r_[np.ones(len(a)), np.zeros(len(b))]
    return pd.Series(
        {f: rs.auc(np.r_[a[f].to_numpy(float), b[f].to_numpy(float)], y) for f in FEAT_NAMES}
    )


def by_stock(fn, *frames):
    """fn applied per stock and pooled: a (feature × stock) frame."""
    out = {}
    for sym in [*SYMBOLS, "all"]:
        parts = [f if sym == "all" else f[f["symbol"] == sym] for f in frames]
        out[sym] = fn(*parts)
    return pd.DataFrame(out)


def heatmaps(panels, title, height=680):
    """One AUC heatmap per (name, frame), features in the order of the first one."""
    order = (panels[0][1]["all"] - 0.5).abs().sort_values(ascending=False).index
    fig = make_subplots(
        rows=1, cols=len(panels), horizontal_spacing=0.1, subplot_titles=[n for n, _ in panels]
    )
    for j, (_, z) in enumerate(panels, start=1):
        z = z.loc[order]
        fig.add_trace(
            go.Heatmap(
                z=z.to_numpy() - 0.5,
                x=list(z.columns),
                y=list(order),
                zmid=0,
                zmin=-0.3,
                zmax=0.3,
                colorscale=plots.DIVERGING,
                showscale=j == len(panels),
                text=z.round(2).to_numpy(),
                texttemplate="%{text}",
                hoverinfo="skip",
            ),
            row=1,
            col=j,
        )
    plots._base(fig, title=title, height=height)
    fig.update_yaxes(autorange="reversed")
    for j in range(2, len(panels) + 1):
        fig.update_yaxes(showticklabels=False, row=1, col=j)
    fig.show()
    return order


at_onset = by_stock(feature_auc, F_on, F_q)
before = by_stock(feature_auc, F_lead, F_ql)
dir_before = by_stock(lambda a: feature_auc(a[a["side"] == "up"], a[a["side"] == "down"]), F_lead)
order3 = heatmaps(
    [
        ("run vs ordinary, at the onset", at_onset),
        (f"run vs ordinary, {LEAD_MIN} min before", before),
        (f"up vs down, {LEAD_MIN} min before ⚠", dir_before),
    ],
    "Fig 3 · each feature's AUC around the onset (0.5 = no information)",
)

## 4 · The confirmation — the moment you could act (Figs 4a, 4b)

Every time a stock closed `CONFIRM_PCT` above its lowest low of the last `LOOKBACK_MIN` minutes (a possible up run)
— or below its highest high (a possible down run) — is a **confirmation**, whether or not a run followed. Its
outcome, from the next bar: **ran** (reached `MOVE_PCT` past that low / high within `WINDOW_MIN` of it), **failed**
(came back through the low / high first) or **stalled** (neither). This is exactly what a live identifier sees, and
the failed confirmations are its natural control.

A trade taken at the confirmation close, with the stop just behind the low and the target at `MOVE_PCT`, risks the
distance back to the low to gain the rest of the run: its **reward ÷ risk** is `r_multiple`, and it breaks even
(before costs) at a run share of `1 / (1 + r)`. Fig 4a sets the run share against that line; Fig 4b asks which
features tell a confirmation that runs from one that fails — per side.

**⚠ Read Fig 4b's run-vs-fail AUCs with care.** A confirmation fires on the first close `CONFIRM_PCT` off the low, but
a big bar can close far beyond that — already most of the way to the target and far from the stop — and such a bar
also has a high recent return and sits high in the day's range. So those features partly measure *where in the
stop–target corridor the entry landed*, which makes "ran" likelier but the remaining gain smaller. The fair read is
**Fig 4d: the trade's actual return** (entry at the confirmation close; exit at the target, at the low as the stop,
or at the close when the window ends — before costs) by fifth of each feature, against one round trip of cost.

In [ ]:
conf = rs.confirmations(
    panel,
    confirm_pct=CONFIRM_PCT,
    lookback_min=LOOKBACK_MIN,
    min_run_pct=MOVE_PCT,
    max_run_min=WINDOW_MIN,
    allow_overnight=ALLOW_OVERNIGHT,
)
conf = conf[(codes[conf["t"]] >= warm) & (bar_no[conf["t"]] * mpb >= SKIP_OPEN_MIN)].reset_index(
    drop=True
)
conf["side"] = np.where(conf["direction"] > 0, "up", "down")
conf["result"] = conf["outcome"].map({1: "ran", -1: "failed", 0: "stalled"})
tab = conf.groupby(["symbol", "side"]).agg(
    confirmations=("t", "size"),
    ran=("outcome", lambda o: (o == 1).mean()),
    failed=("outcome", lambda o: (o == -1).mean()),
    stalled=("outcome", lambda o: (o == 0).mean()),
    median_r=("r_multiple", "median"),
)
tab["break-even ran share"] = 1 / (1 + tab["median_r"])
tab["ran − break-even"] = tab["ran"] - tab["break-even ran share"]
fees, slip = bt.strategy_costs(strat, panel, settings)
RT = 2 * (fees + slip)  # one round trip at the real order size
tab["mean trade bp"] = 1e4 * conf.groupby(["symbol", "side"])["ret"].mean()
tab["after costs bp"] = tab["mean trade bp"] - 1e4 * RT
display(tab.round(3))

fig = go.Figure()
for side, colr in (("up", plots._LONG), ("down", plots._SHORT)):
    t_ = tab.xs(side, level="side")
    fig.add_trace(
        go.Scatter(
            x=t_["break-even ran share"],
            y=t_["ran"],
            mode="markers+text",
            name=side,
            text=list(t_.index),
            textposition="top center",
            marker={"size": 12, "color": colr},
        )
    )
fig.add_trace(
    go.Scatter(
        x=[0, 0.6],
        y=[0, 0.6],
        mode="lines",
        name="break-even (before costs)",
        line={"color": plots._MUTED, "dash": "dash"},
    )
)
plots._base(
    fig,
    title="Fig 4a · confirmations: share that ran vs the share needed to break even",
    height=420,
)
below(fig, -0.15)
fig.update_xaxes(title_text="needed: 1 / (1 + reward ÷ risk)", tickformat=".0%")
fig.update_yaxes(title_text="ran", tickformat=".0%")
fig.show()

F_c = rs.at_points(feats, conf["t"], conf["col"]).assign(
    symbol=conf["symbol"], side=conf["side"], result=conf["result"]
)
ran_vs_failed = {}
for side in ("up", "down"):
    d = F_c[F_c["side"] == side]
    ran_vs_failed[side] = by_stock(
        lambda a: feature_auc(a[a["result"] == "ran"], a[a["result"] == "failed"]), d
    )
order4 = heatmaps(
    [
        ("up confirmations: ran vs failed", ran_vs_failed["up"]),
        ("down confirmations: ran vs failed", ran_vs_failed["down"]),
    ],
    "Fig 4b · which features tell a confirmation that runs from one that fails (0.5 = none)",
)

# Fig 4d — the fair read: the trade's return by fifth of each of the strongest features
top4 = list(
    pd.concat([(ran_vs_failed[s]["all"] - 0.5).abs() for s in ("up", "down")], axis=1)
    .max(axis=1)
    .sort_values(ascending=False)
    .index[:4]
)
fig = make_subplots(rows=1, cols=len(top4), subplot_titles=top4, shared_yaxes=True)
for j, f in enumerate(top4, start=1):
    for side, colr in (("up", plots._LONG), ("down", plots._SHORT)):
        d = F_c[F_c["side"] == side].assign(ret=conf.loc[conf["side"] == side, "ret"].to_numpy())
        d = d[np.isfinite(d[f])]
        fifth = pd.qcut(d[f], 5, labels=False, duplicates="drop")
        m = 1e4 * d.groupby(fifth)["ret"].mean()
        fig.add_trace(
            go.Scatter(
                x=[f"Q{i + 1}" for i in m.index],
                y=m.to_numpy(),
                mode="lines+markers",
                name=side,
                showlegend=j == 1,
                line={"color": colr},
            ),
            row=1,
            col=j,
        )
    fig.add_trace(
        go.Scatter(
            x=["Q1", "Q5"],
            y=[1e4 * RT] * 2,
            mode="lines",
            name="one round trip of cost",
            showlegend=j == 1,
            line={"color": plots._LOSS, "dash": "dot", "width": 1},
        ),
        row=1,
        col=j,
    )
    fig.add_trace(
        go.Scatter(
            x=["Q1", "Q5"],
            y=[0, 0],
            mode="lines",
            showlegend=False,
            line={"color": plots._MUTED, "width": 1},
        ),
        row=1,
        col=j,
    )
plots._base(
    fig,
    title="Fig 4d · the trade from the confirmation: mean return (bp, before costs) by fifth "
    "of the feature",
    height=400,
)
below(fig, -0.15)
fig.show()

### 4c · The strongest confirmation features (Fig 4c)

The six features that best separate *ran* from *failed* (pooled over both sides), as distributions per side.

In [ ]:
strength = pd.concat([(ran_vs_failed[s]["all"] - 0.5).abs() for s in ("up", "down")], axis=1).max(
    axis=1
)
top = list(strength.sort_values(ascending=False).index[:6])
fig = make_subplots(rows=2, cols=3, subplot_titles=top, vertical_spacing=0.14)
for k, f in enumerate(top):
    rr, cc = k // 3 + 1, k % 3 + 1
    for res_, colr in (("ran", plots._WIN), ("failed", plots._LOSS), ("stalled", plots._MUTED)):
        d = F_c[F_c["result"] == res_]
        fig.add_trace(
            go.Box(
                x=d["side"],
                y=d[f],
                name=res_,
                marker_color=colr,
                boxpoints=False,
                showlegend=k == 0,
                legendgroup=res_,
            ),
            row=rr,
            col=cc,
        )
fig.update_layout(boxmode="group")
plots._base(fig, title="Fig 4c · confirmation features by outcome (x = side)", height=640)
below(fig, -0.08)
fig.show()

## 5 · The build-up (Fig 5)

Median volume and range of each bar ÷ their normal for the clock time, from `PRE_MIN` minutes before to `POST_MIN`
after — **top** aligned on the onset (runs vs ordinary moments), **bottom** aligned on the confirmation (confirmations
that ran vs those that failed). If a run announces itself, the lines rise before 0.

In [ ]:
pre_b, post_b = S["_bars"](PRE_MIN, panel.bar), S["_bars"](POST_MIN, panel.bar)
offsets = np.arange(-pre_b, post_b + 1)


def curve(frame, starts, cols):
    rows_ = np.asarray(starts)[:, None] + offsets[None, :]
    safe = np.clip(rows_, 0, n_bars - 1)
    vals = frame[safe, np.asarray(cols)[:, None]]
    keep = (rows_ >= 0) & (rows_ < n_bars) & (codes[safe] == codes[np.asarray(starts)][:, None])
    return np.nanmedian(np.where(keep, vals, np.nan), axis=0)


groups = [
    (
        "onset",
        {
            "up run": runs[runs["side"] == "up"],
            "down run": runs[runs["side"] == "down"],
            "ordinary": q.assign(onset=q["start"]),
        },
        "onset",
    ),
    (
        "confirmation",
        {"ran": conf[conf["outcome"] == 1], "failed": conf[conf["outcome"] == -1]},
        "t",
    ),
]
COL = {
    "up run": plots._LONG,
    "down run": plots._SHORT,
    "ordinary": plots._MUTED,
    "ran": plots._WIN,
    "failed": plots._LOSS,
}
fig = make_subplots(
    rows=2,
    cols=2,
    vertical_spacing=0.16,
    subplot_titles=(
        "aligned on the onset: volume ÷ normal",
        "aligned on the onset: range ÷ normal",
        "aligned on the confirmation: volume ÷ normal",
        "aligned on the confirmation: range ÷ normal",
    ),
)
for i, (_, gset, key) in enumerate(groups, start=1):
    for j, f in enumerate(("bar_vol_vs_tod", "bar_range_vs_tod"), start=1):
        a = feats[f].to_numpy(float)
        for name, g in gset.items():
            fig.add_trace(
                go.Scatter(
                    x=offsets * mpb,
                    y=curve(a, g[key], g["col"]),
                    mode="lines",
                    name=name,
                    showlegend=j == 1,
                    line={"color": COL[name], "width": 2},
                ),
                row=i,
                col=j,
            )
        fig.add_trace(
            go.Scatter(
                x=[offsets[0] * mpb, offsets[-1] * mpb],
                y=[1, 1],
                mode="lines",
                name="normal",
                showlegend=(i, j) == (1, 1),
                line={"color": plots._INK, "dash": "dot", "width": 1},
            ),
            row=i,
            col=j,
        )
        fig.add_vline(x=0, line={"color": plots._INK, "dash": "dash", "width": 1}, row=i, col=j)
plots._base(
    fig,
    title="Fig 5 · median volume and range, minutes around the onset and the confirmation",
    height=640,
)
below(fig, -0.08)
fig.show()

## 6 · Run cards (Figs 6.x)

For each stock, `RUNS_PER_STOCK` runs (half up, half down, drawn with `SEED`; or the numbers in `RUN_PICK`), each
drawn whole, from `CARD_PRE_MIN` minutes before the onset (the previous session when that reaches back past the open)
to `POST_MIN` after the run's end:

* **top** — the candles; the shaded strip is the `PRE_MIN` minutes before the onset (the feature window);
  ● the **onset** (the swing low / high), ▲/▼ the **confirmation** (the first close `CONFIRM_PCT` past it), ◆ the bar
  that reached `MOVE_PCT`, ★ the **end** of the run; dotted lines: the onset level, the confirmation level and the
  `MOVE_PCT` target;
* **middle** — each bar's volume, with the stock's normal volume at that clock time;
* **bottom** — each bar's range ÷ its normal (1 = a normal bar).

The feature values at the onset and at the confirmation are listed under each card. Times are New York.

In [ ]:
vol_norm = bt.volume_profile(panel.volume, tz, days=PROFILE_DAYS, stat=PROFILE_STAT).to_numpy(float)
rng_ratio = feats["bar_range_vs_tod"].to_numpy(float)
card_pre_b = S["_bars"](CARD_PRE_MIN, panel.bar)


def pick_runs(sym):
    r = runs[runs["symbol"] == sym]
    if sym in RUN_PICK:
        return r[r["run"].isin(RUN_PICK[sym])]
    half = max(1, RUNS_PER_STOCK // 2)
    parts = [
        r[r["side"] == s].sample(min(half, int((r["side"] == s).sum())), random_state=SEED)
        for s in ("up", "down")
    ]
    return pd.concat(parts).sort_values("onset")


def ts(i):
    return panel.close.index[i].tz_convert(tz).tz_localize(None)


def card(r):
    col, sgn = r.col, r.direction
    a, b = max(r.onset - card_pre_b, 0), min(r.peak + post_b, n_bars - 1)
    span = np.arange(a, b + 1)
    x = panel.close.index[span].tz_convert(tz).tz_localize(None)
    colr = plots._LONG if sgn > 0 else plots._SHORT
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True, row_heights=[0.56, 0.24, 0.2], vertical_spacing=0.04
    )
    fig.add_trace(
        go.Candlestick(
            x=x,
            open=o_[span, col],
            high=h_[span, col],
            low=l_[span, col],
            close=c_[span, col],
            showlegend=False,
            increasing_line_color="#a9c6ba",
            decreasing_line_color="#e0b3b0",
        ),
        row=1,
        col=1,
    )
    levels = (
        (r.base, "onset level", "dot"),
        (r.base * (1 + sgn * CONFIRM_PCT), "confirmation level", "dashdot"),
        (r.base * (1 + sgn * MOVE_PCT), f"{MOVE_PCT:.0%} target", "dash"),
    )
    for y, name, dash in levels:
        fig.add_trace(
            go.Scatter(
                x=[ts(r.onset), ts(r.peak)],
                y=[y, y],
                mode="lines",
                name=name,
                line={"color": colr, "dash": dash, "width": 1.2},
            ),
            row=1,
            col=1,
        )
    ext = h_[r.peak, col] if sgn > 0 else l_[r.peak, col]
    hit_px = r.base * (1 + sgn * MOVE_PCT)
    marks = (
        (r.onset, r.base, "circle", "onset"),
        (
            r.confirm,
            c_[r.confirm, col],
            "triangle-up" if sgn > 0 else "triangle-down",
            "confirmation",
        ),
        (r.hit, hit_px, "diamond", f"{MOVE_PCT:.0%} reached"),
        (r.peak, ext, "star", "end of the run"),
    )
    for i, y, sym_, name in marks:
        fig.add_trace(
            go.Scatter(
                x=[ts(i)],
                y=[y],
                mode="markers",
                name=name,
                marker={
                    "symbol": sym_,
                    "size": 12,
                    "color": colr,
                    "line": {"color": plots._INK, "width": 1},
                },
            ),
            row=1,
            col=1,
        )
    fig.add_trace(go.Bar(x=x, y=v_[span, col], name="volume", marker_color="#cfcdc4"), row=2, col=1)
    fig.add_trace(
        go.Scatter(
            x=x,
            y=vol_norm[span, col],
            mode="lines",
            name=f"normal volume ({PROFILE_DAYS}-day {PROFILE_STAT})",
            line={"color": plots.SERIES[4], "width": 1.5},
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Bar(x=x, y=rng_ratio[span, col], name="range ÷ normal", marker_color=plots.SERIES[2]),
        row=3,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[x[0], x[-1]],
            y=[1, 1],
            mode="lines",
            name="normal range",
            line={"color": plots._INK, "dash": "dot", "width": 1},
        ),
        row=3,
        col=1,
    )
    x0 = ts(max(r.onset - pre_b, 0))
    for row in (1, 2, 3):  # shapes AFTER their subplot's traces
        fig.add_vrect(
            x0=x0,
            x1=ts(r.onset),
            fillcolor=colr,
            opacity=0.08,
            line_width=0,
            layer="below",
            row=row,
            col=1,
        )
    title = (
        f"Fig 6 · {r.symbol} run #{r.run}: {r.side} {r.size:.1%} over {r.duration_min:.0f} min "
        f"(onset {r.local:%a %Y-%m-%d %H:%M}; confirmed +{r.to_confirm_min:.0f} min; "
        f"{MOVE_PCT:.0%} at +{r.to_hit_min:.0f} min)"
    )
    plots._base(fig, title=title, height=640)
    below(fig, -0.06)
    fig.update_xaxes(
        rangeslider_visible=False, rangebreaks=plots._intraday_breaks(pd.DatetimeIndex(x))
    )
    fig.update_yaxes(title_text="price", row=1, col=1)
    fig.update_yaxes(title_text="volume", row=2, col=1)
    fig.update_yaxes(title_text="range ÷ normal", row=3, col=1)
    return fig


shown = []
for sym in SYMBOLS:
    for r in pick_runs(sym).itertuples():
        card(r).show()
        vals = rs.at_points(feats, [r.onset, r.confirm], [r.col, r.col])
        vals.index = ["at the onset", "at the confirmation"]
        display(vals.round(3))
        shown.append(f"{sym}#{r.run}")

## 7 · Checks

In [ ]:
moved = []
for r in runs_all.itertuples():
    ext = h_[r.hit, r.col] if r.direction > 0 else l_[r.hit, r.col]
    moved.append(r.direction * (ext / r.base - 1) >= MOVE_PCT - 1e-12)
fast = bool((runs_all["to_hit_min"] <= WINDOW_MIN).all())
ordered = bool(
    ((runs_all["onset"] <= runs_all["confirm"]) & (runs_all["hit"] <= runs_all["peak"])).all()
)
no_overlap = all(
    (g["onset"].to_numpy()[1:] >= g["peak"].to_numpy()[:-1]).all()
    for _, g in runs_all.groupby("symbol")
)
causal = bool(
    (
        (conf["ref"] <= conf["t"]) & (conf["t"] - conf["ref"] < S["_bars"](LOOKBACK_MIN, panel.bar))
    ).all()
)
same_day = ALLOW_OVERNIGHT or bool((codes[runs_all["onset"]] == codes[runs_all["peak"]]).all())
checks = rs.checks_table(
    [
        ("every run moved MOVE_PCT from its onset", f"{np.mean(moved):.0%}", "100%", all(moved)),
        ("every run reached MOVE_PCT within WINDOW_MIN", fast, True, fast),
        ("onset ≤ confirmation, MOVE_PCT ≤ end, in every run", ordered, True, ordered),
        ("runs never overlap", no_overlap, True, no_overlap),
        (
            "confirmations only look back (reference ≤ bar, inside LOOKBACK_MIN)",
            causal,
            True,
            causal,
        ),
        ("no run crosses the close (ALLOW_OVERNIGHT off)", same_day, True, same_day),
    ]
)
display(checks)
rs.scorecard(
    {
        "notebook": "move_explorer",
        "symbols": SYMBOLS,
        "move_pct": MOVE_PCT,
        "window_min": WINDOW_MIN,
        "reversal_pct": REVERSAL_PCT,
        "confirm_pct": CONFIRM_PCT,
        "bar_min": BAR_MIN,
        "runs": {s: int((runs_all["symbol"] == s).sum()) for s in SYMBOLS},
        "old_slices_median_already_moved": float(already.median()),
        "confirmations": {
            f"{s}|{d}": {
                "n": int(tab.loc[(s, d), "confirmations"]),
                "ran": tab.loc[(s, d), "ran"],
                "needed": tab.loc[(s, d), "break-even ran share"],
            }
            for s, d in tab.index
        },
        "best_ran_vs_failed_strength": {
            side: float((ran_vs_failed[side]["all"] - 0.5).abs().max()) for side in ("up", "down")
        },
        "mean_trade_bp": {f"{s}|{d}": tab.loc[(s, d), "mean trade bp"] for s, d in tab.index},
        "round_trip_bp": 1e4 * RT,
        "cards": shown,
    }
)